# Cosmic Web $\chi$-Field Structure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/notebooks/LFM_Cosmic_Web_Chi.ipynb)

## What This Notebook Demonstrates

In LFM, matter density sources $\chi$-wells via GOV-02. Cosmic filaments should have **lower $\chi$** than voids.

- Synthetic cosmic web via FFT power spectrum
- Solve $\nabla^2 \chi = (\kappa/c^2)(\rho - \rho_{\text{mean}})$ with FFT
- Classify structures (voids, filaments, clusters) and measure $\chi$ in each

**Parameters**: $\chi_0 = 19$, $\kappa = 1/63$, box = 100 Mpc/h, grid = $64^3$

---

In [ ]:
import numpy as np
from scipy.fft import fftn, ifftn
import matplotlib.pyplot as plt

# LFM parameters
chi0 = 19.0
kappa = 1/63
box_size = 100.0  # Mpc/h
n_grid = 64  # Reduced for Colab speed
dx = box_size / n_grid

# Generate synthetic cosmic web
np.random.seed(42)
kx = np.fft.fftfreq(n_grid, d=dx) * 2 * np.pi
ky = np.fft.fftfreq(n_grid, d=dx) * 2 * np.pi
kz = np.fft.fftfreq(n_grid, d=dx) * 2 * np.pi
KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
K = np.sqrt(KX**2 + KY**2 + KZ**2)
K[0, 0, 0] = 1

P_k = K**0.96 / (1 + (K / 0.01)**2)**2
P_k[0, 0, 0] = 0
phases = np.random.uniform(0, 2 * np.pi, (n_grid, n_grid, n_grid))
density = np.real(ifftn(np.sqrt(P_k) * np.exp(1j * phases)))
density = density / np.std(density) * 0.5
density = np.exp(density)
density = density / np.mean(density)

# Solve GOV-02 for chi field
K2 = KX**2 + KY**2 + KZ**2
K2[0, 0, 0] = 1
source = kappa * (density - 1.0)
chi_dev = np.real(ifftn(-fftn(source) / K2))
chi = chi0 + chi_dev

# Classify structures
clusters = density > 2.0
voids = density < 0.3
filaments = (~clusters) & (~voids)

print('=' * 60)
print('COSMIC WEB CHI-FIELD RESULTS')
print('=' * 60)
for name, mask in [('Voids', voids), ('Filaments', filaments), ('Clusters', clusters)]:
    vals = chi[mask]
    print(f'  {name:10s}: chi = {np.mean(vals):.6f} +/- {np.std(vals):.6f}  ({np.sum(mask)/mask.size*100:.1f}% vol)')

ratio = np.mean(chi[filaments]) / np.mean(chi[voids])
print(f'\n  chi_filament / chi_void = {ratio:.6f}')
print(f'  H0 REJECTED: {ratio < 0.999}')
if ratio < 0.999:
    print(f'  LFM predicts measurable chi variation in cosmic web!')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
mid = n_grid // 2

ax = axes[0]
im = ax.imshow(np.log10(density[:, :, mid]), cmap='viridis', origin='lower')
ax.set_title('Log Density')
plt.colorbar(im, ax=ax)

ax = axes[1]
im = ax.imshow(chi[:, :, mid], cmap='RdBu_r', origin='lower')
ax.set_title('chi Field')
plt.colorbar(im, ax=ax)

ax = axes[2]
for name, mask, color in [('Voids', voids, 'blue'), ('Filaments', filaments, 'green'), ('Clusters', clusters, 'red')]:
    ax.hist(chi[mask].flatten(), bins=50, alpha=0.5, label=name, color=color, density=True)
ax.axvline(chi0, color='k', ls='--', label='chi_0 = 19')
ax.set_xlabel('chi')
ax.set_title('chi Distribution by Structure')
ax.legend()

plt.suptitle('LFM Cosmic Web chi-Field Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Key Result

- Clusters have **lower** $\chi$ (deeper potential wells)
- Voids have $\chi$ slightly above $\chi_0$
- The $\chi$ variation is a direct prediction of GOV-02
- Observable via light travel time differences between filament and void paths